<a href="https://colab.research.google.com/github/Lornatt88/maai902-nlp-solution/blob/main/notebooks/05_rag_explanations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5: RAG explanations
Explains each flagged clause in plain English: the system first retrieves expert-written legal explanations, then an LLM writes the explanation using only those.

## 1. Building the knowledge base
Loads 79 expert explanations from Ruggeri et al. (2022) for five unfair types, each with a severity level. Adds three explanations for the remaining types (Jurisdiction, Choice of law, Contract by using), based on Lippi et al. (2019), Section 3.1.

In [ ]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/federicoruggeri/Memnet_ToS/master/local_database/KB"
CODES = {"A": "Arbitration", "CH": "Unilateral change", "CR": "Content removal",
         "LTD": "Limitation of liability", "TER": "Unilateral termination"}

# Ruggeri et al. (2022) explanations
parts = []
for code, type_name in CODES.items():
    t = pd.read_csv(f"{BASE}/{code}_explanations.csv", index_col=0)
    t = t.rename(columns={"Id": "name", "Tag": "tag", "Explanation": "text"})
    t["id"] = t.index                     # keep each explanation's number (used by the expert links)
    t["type"] = type_name
    t["severity"] = t["tag"].str[-1].map({"2": "potentially unfair", "3": "clearly unfair"})
    t["source"] = "Ruggeri et al. (2022)"
    parts.append(t)

# Own explanations for the 3 missing types, based on Lippi et al. (2019), Section 3.1
own = pd.DataFrame([
    {"type": "Jurisdiction", "name": "court_away_from_residence", "severity": "clearly unfair",
     "text": "the clause requires any court case to take place away from the consumer's place of residence, in a different city or country, making it harder and more expensive to take legal action.",
     "source": "own, based on Lippi et al. (2019), Section 3.1 (ECJ case C-240/98)"},
    {"type": "Choice of law", "name": "law_not_of_residence", "severity": "potentially unfair",
     "text": "the clause makes a law other than the law of the consumer's country of residence govern the contract, which may reduce the legal protection the consumer would normally have.",
     "source": "own, based on Lippi et al. (2019), Section 3.1"},
    {"type": "Contract by using", "name": "use_is_consent", "severity": "potentially unfair",
     "text": "the clause binds the consumer to the terms simply by using the service, without requiring them to confirm that they have read and accepted the terms.",
     "source": "own, based on Lippi et al. (2019), Section 3.1"},
])

kb = pd.concat(parts + [own], ignore_index=True)
kb["text"] = (kb["text"].str.strip()
              .str.replace(r"^since\s+(the clause states that\s+)?", "", regex=True, case=False))
print("Explanations:", len(kb))
print(kb.groupby("type").size())

Explanations: 87
type
Arbitration                 8
Choice of law               1
Content removal            17
Contract by using           1
Jurisdiction                1
Limitation of liability    24
Unilateral change           8
Unilateral termination     27
dtype: int64


The knowledge base contains 87 explanations: 84 from the publicly released files of Ruggeri et al. (2022) for five unfair types (the paper reports 79, so the released files appear to be a slightly updated version), and 3 written for this project for Jurisdiction, Choice of law and Contract by using, based on the definitions and fairness rules in Lippi et al. (2019), Section 3.1.



---



## 2. Retrieval
Turns every explanation into an embedding (a list of numbers representing its meaning) with all-MiniLM-L6-v2, a model built for comparing sentence meanings. For a flagged clause, it compares the clause's embedding with the explanations of the same unfair type and returns the 3 closest.

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
kb_vecs = embedder.encode(kb["text"].tolist(), normalize_embeddings=True)   # one embedding per explanation

def retrieve(clause, clause_type, k=3):
    """Return the k explanations of this type that are closest in meaning to the clause."""
    mask = (kb["type"] == clause_type).values                # only explanations of this type
    q = embedder.encode([clause], normalize_embeddings=True)[0]
    scores = kb_vecs[mask] @ q                               # closeness in meaning (cosine similarity)
    top = np.argsort(scores)[::-1][:k]                       # highest scores first
    out = kb[mask].iloc[top].copy()
    out["score"] = scores[top].round(2)
    return out[["name", "severity", "score", "text"]]

# Test on a real termination clause from the dataset
clause = "tinder may terminate your account at any time without notice if it believes that you have violated this agreement ."
print(retrieve(clause, "Unilateral termination").to_string())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

             name            severity  score                                                                                                                                                                                                text
64        dormant  potentially unfair   0.48                                                                                             the contract or access may be terminated where the account has been left dormant for a prescribed time.
74         sex_of  potentially unfair   0.47  the contract or access may be terminated where the user is a registered sex offender, or engaged or attempted to engage in sexual conduct with minors, or has been involved with child pornography
63  cred_security  potentially unfair   0.47                                                    the contract or access may be terminated where the user fails to maintain the security of the login credentials and/or a security breach occurs.


## 3. Testing the retrieval
Ruggeri et al. (2022) linked each unfair clause in their dataset to the expert explanation(s) that apply to it. These links are used as the correct answers to measure how often the retrieval finds the right explanation.

### 3a. Looking at the expert links
Loads Ruggeri's dataset and shows how the links are stored.

In [ ]:
url = "https://raw.githubusercontent.com/federicoruggeri/Memnet_ToS/master/local_database/ToS_100/dataset.csv"
rug = pd.read_csv(url)
print("Rows:", len(rug))
print("Columns:", list(rug.columns), "\n")

targets = [c for c in rug.columns if c.endswith("_targets")]
for c in targets:
    filled = rug[c].dropna()
    filled = filled[filled.astype(str).str.strip().isin(["", "[]"]) == False]
    print(f"{c}: {len(filled)} clauses with links | examples: {filled.head(3).tolist()}")

Rows: 20417
Columns: ['Unnamed: 0', 'A', 'CH', 'CR', 'J', 'LAW', 'LTD', 'PINC', 'TER', 'USE', 'document', 'document_ID', 'label', 'text', 'TER_targets', 'LTD_targets', 'A_targets', 'CH_targets', 'CR_targets'] 

TER_targets: 420 clauses with links | examples: ['[1, 0, 19, 25]', '[20]', '[23, 27]']
LTD_targets: 626 clauses with links | examples: ['[0, 8, 13, 3, 2, 11, 1]', '[4]', '[0]']
A_targets: 106 clauses with links | examples: ['[1, 0]', '[0, 4, 1]', '[1]']
CH_targets: 344 clauses with links | examples: ['[0, 6]', '[0, 2]', '[0]']
CR_targets: 216 clauses with links | examples: ['[4, 5]', '[4]', '[9]']


### 3b. Comparing retrieval models
For each of the 1,712 clauses linked by experts, retrieves the top 3 explanations of its type and checks whether the first pick (top-1) or any of the three picks (top-3) is one of the correct explanations. Three sentence-embedding models are compared, together with random guessing as a reference point.

In [ ]:
import ast
from math import comb

def gold_rows(code):
    """Clauses that experts linked to explanations of this type, with their correct explanation numbers."""
    col = f"{code}_targets"
    rows = rug[rug[col].notna() & (rug[col].astype(str).str.strip() != "[]")]
    return rows["text"].tolist(), [set(ast.literal_eval(str(x))) for x in rows[col]]

def evaluate(model_name):
    emb = SentenceTransformer(model_name)
    vecs = emb.encode(kb["text"].tolist(), normalize_embeddings=True)
    hit1 = hit3 = rand1 = rand3 = total = 0
    for code, type_name in CODES.items():
        mask = (kb["type"] == type_name).values
        ids = kb.loc[mask, "id"].values
        texts, golds = gold_rows(code)
        scores = emb.encode(texts, normalize_embeddings=True, batch_size=64) @ vecs[mask].T
        order = np.argsort(-scores, axis=1)
        n = mask.sum()
        for r, gold in enumerate(golds):
            top = ids[order[r]]
            hit1 += top[0] in gold
            hit3 += bool(gold & set(top[:3]))
            g = len(gold & set(ids))
            rand1 += g / n                                   # chance of a random first pick being right
            rand3 += 1 - comb(n - g, 3) / comb(n, 3)         # chance of 3 random picks including a right one
            total += 1
    return 100 * hit1 / total, 100 * hit3 / total, 100 * rand1 / total, 100 * rand3 / total

for m in ["sentence-transformers/all-MiniLM-L6-v2",
          "sentence-transformers/all-mpnet-base-v2",
          "BAAI/bge-base-en-v1.5"]:
    t1, t3, r1, r3 = evaluate(m)
    print(f"{m:<42} top-1 {t1:5.1f}% | top-3 {t3:5.1f}%")
print(f"{'Random guess':<42} top-1 {r1:5.1f}% | top-3 {r3:5.1f}%")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

sentence-transformers/all-MiniLM-L6-v2     top-1  33.5% | top-3  61.9%


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BAAI/bge-base-en-v1.5                      top-1  35.4% | top-3  61.7%
Random guess                               top-1  11.7% | top-3  31.0%


On 1,712 clauses linked to explanations by Ruggeri et al. (2022), MiniLM found a correct explanation in its top 3 for 61.9% of clauses (33.5% on the first pick), against 31.0% for random guessing. BGE performed similarly (61.7% / 35.4%) and MPNet worse (56.4% / 28.3%). MiniLM was kept because it matched the best top-3 result while being about five times smaller, which keeps the demo fast without a GPU.



---



### 3c. Comparing only the distinctive words
Within each type, words that appear in at least 70% of the explanations are removed before comparing, so the retrieval focuses on what makes each explanation different. The full explanation text is kept for the LLM. MiniLM is tested again on the same 1,712 clauses.

In [ ]:
from collections import Counter

def distinctive(texts, cutoff=0.7):
    """Remove words that appear in at least 70% of the texts."""
    docs = [set(t.lower().split()) for t in texts]
    counts = Counter(w for d in docs for w in d)
    common = {w for w, c in counts.items() if c / len(texts) >= cutoff}
    return [" ".join(w for w in t.split() if w.lower() not in common) for t in texts]

kb["core"] = kb["text"]
for type_name, g in kb.groupby("type"):
    if len(g) >= 5:                                      # only types with several explanations
        kb.loc[g.index, "core"] = distinctive(g["text"].tolist())

print("Example (breach):", kb.loc[kb["name"] == "breach", "core"].iloc[0][:120], "\n")

def evaluate(model_name, kb_col="text"):
    emb = SentenceTransformer(model_name)
    vecs = emb.encode(kb[kb_col].tolist(), normalize_embeddings=True)
    hit1 = hit3 = total = 0
    for code, type_name in CODES.items():
        mask = (kb["type"] == type_name).values
        ids = kb.loc[mask, "id"].values
        texts, golds = gold_rows(code)
        scores = emb.encode(texts, normalize_embeddings=True, batch_size=64) @ vecs[mask].T
        order = np.argsort(-scores, axis=1)
        for r, gold in enumerate(golds):
            top = ids[order[r]]
            hit1 += top[0] in gold
            hit3 += bool(gold & set(top[:3]))
            total += 1
    return 100 * hit1 / total, 100 * hit3 / total

t1, t3 = evaluate("sentence-transformers/all-MiniLM-L6-v2", kb_col="core")
print(f"MiniLM, distinctive words only:  top-1 {t1:.1f}% | top-3 {t3:.1f}%")
print("MiniLM, full text (from 3b):     top-1 33.5% | top-3 61.9%")

Example (breach): can where user fails to adhere to its terms, community standards, spirit of ToS community terms, including inappropriate 



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MiniLM, distinctive words only:  top-1 28.6% | top-3 57.2%
MiniLM, full text (from 3b):     top-1 33.5% | top-3 61.9%


### ***Rejected***

Removing the words shared by most explanations of a type lowered retrieval accuracy (top-3 57.2% vs 61.9%), probably because the shortened explanations were no longer full sentences. The full explanation text is therefore used.



---



### 3d. Adding a reranker
MiniLM's 30 closest (all of the type) explanations, then a cross-encoder (ms-marco-MiniLM-L-6-v2) reads the clause and each explanation together and re-orders them. Tested on the same 1,712 clauses.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vecs = emb.encode(kb["text"].tolist(), normalize_embeddings=True)

hit1 = hit3 = total = 0
for code, type_name in CODES.items():
    mask = (kb["type"] == type_name).values
    ids, kb_texts = kb.loc[mask, "id"].values, kb.loc[mask, "text"].values
    texts, golds = gold_rows(code)
    scores = emb.encode(texts, normalize_embeddings=True, batch_size=64) @ vecs[mask].T
    first10 = np.argsort(-scores, axis=1)[:, :30]                     # MiniLM's 30 closest
    for r, (clause, gold) in enumerate(zip(texts, golds)):
        cand = first10[r]
        new_scores = reranker.predict([(clause, kb_texts[c]) for c in cand])   # read together
        top = ids[cand[np.argsort(-new_scores)]]                     # re-ordered
        hit1 += top[0] in gold
        hit3 += bool(gold & set(top[:3]))
        total += 1

print(f"MiniLM + reranker:  top-1 {100 * hit1 / total:.1f}% | top-3 {100 * hit3 / total:.1f}%")
print("MiniLM alone (3b):  top-1 33.5% | top-3 61.9%")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MiniLM + reranker:  top-1 43.6% | top-3 67.8%
MiniLM alone (3b):  top-1 33.5% | top-3 61.9%


Adding a cross-encoder reranker (ms-marco-MiniLM-L-6-v2) to re-order MiniLM’s 10 closest explanations raised top-1 accuracy from 33.5% to 43.8% and top-3 accuracy from 61.9% to 67.0%. The reranker is therefore used in the final retrieval.

Re-ranking all explanations of the type instead of a shortlist of 10 gave almost identical results (top-3 67.8% vs 67.0%), so all explanations are re-ranked.



---



### 3e. Final retrieval
The retrieval used from now on: MiniLM shortlists all explanations of the clause's type, then the reranker re-orders them and the top 3 are returned.

In [ ]:
def retrieve(clause, clause_type, k=3, shortlist=30):
    """MiniLM shortlists all explanations of this type, the reranker re-orders them, and the top k are returned."""
    mask = (kb["type"] == clause_type).values
    sub = kb[mask]
    q = embedder.encode([clause], normalize_embeddings=True)[0]
    first = np.argsort(-(kb_vecs[mask] @ q))[:shortlist]                  # step 1: shortlist
    rs = reranker.predict([(clause, sub["text"].iloc[c]) for c in first])  # step 2: read side by side
    order = np.argsort(-rs)[:k]
    out = sub.iloc[first[order]].copy()
    out["score"] = rs[order].round(2)
    return out[["name", "severity", "score", "text"]]

# Same test as Step 2
clause = "tinder may terminate your account at any time without notice if it believes that you have violated this agreement ."
print(retrieve(clause, "Unilateral termination").to_string())

                    name            severity  score                                                                                                                                                    text
70            over_quota  potentially unfair  -0.41                            the contract or access may be terminated where the user has been violating the time, storage or other limits of the service.
64               dormant  potentially unfair  -1.14                                                 the contract or access may be terminated where the account has been left dormant for a prescribed time.
69  gen_rights_violation  potentially unfair  -1.44  the clause generally states the contract or access may be terminated where the user has violated the rights of the service provider or other entities.




---



## 4. Writing the explanation
Loads Qwen2.5-7B-Instruct (4-bit). For a flagged clause, the 3 closest expert explanations are retrieved, and Qwen writes a short plain-English explanation and practical advice using only those explanations, so the answer stays grounded in expert sources.

In [ ]:
!pip install -q bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

llm_name = "Qwen/Qwen2.5-7B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)

def ask_llm(prompt, max_new_tokens=250):
    """Send one message to the LLM and return its answer."""
    messages = [{"role": "user", "content": prompt}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

EXPLAIN_PROMPT = """You help everyday users understand unfair clauses in online Terms of Service.

Clause: {clause}
Type: {clause_type}

Expert explanations (use ONLY these, do not add other legal facts):
{sources}

Write:
1. "Why this may be unfair:" 2 short sentences in plain English, no legal jargon.
2. "What you can do:" 1 or 2 short, practical tips.
Do not give legal advice and do not mention the explanation numbers."""

def explain(clause, clause_type):
    """Retrieve 3 expert explanations, then let the LLM write a plain-English explanation from them."""
    found = retrieve(clause, clause_type, k=3)
    sources = "\n".join(f"- {t}" for t in found["text"])
    answer = ask_llm(EXPLAIN_PROMPT.format(clause=clause, clause_type=clause_type, sources=sources))
    return answer, found

# Test on 3 clauses of different types
tests = [
    ("tinder may terminate your account at any time without notice if it believes that you have violated this agreement .", "Unilateral termination"),
    ("any dispute arising out of these terms shall be resolved by binding arbitration , and you waive your right to a jury trial .", "Arbitration"),
    ("any judicial proceeding relating to these terms will be brought in the courts of san francisco county , california .", "Jurisdiction"),
]
for clause, ctype in tests:
    answer, found = explain(clause, ctype)
    print(f"===== {ctype} =====\nCLAUSE: {clause}\n\n{answer}\n")
    print("Sources:", ", ".join(f"{n} ({s})" for n, s in zip(found["name"], found["severity"])), "\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.4 MB/s eta 0:00:00


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

===== Unilateral termination =====
CLAUSE: tinder may terminate your account at any time without notice if it believes that you have violated this agreement .

1. "This clause might be unfair because it allows Tinder to end your account without giving you a reason or chance to fix the issue."
2. "Keep good records of your interactions and communications on Tinder, and review the terms regularly to stay informed about any changes."

Sources: dormant (potentially unfair), sex_of (potentially unfair), cred_security (potentially unfair) 

===== Arbitration =====
CLAUSE: any dispute arising out of these terms shall be resolved by binding arbitration , and you waive your right to a jury trial .

1. This clause may be unfair because it forces you to resolve disputes through arbitration rather than a court, potentially limiting your ability to have a case heard by a jury.
2. To address this, consider seeking legal advice to understand the implications of arbitration and whether there are ways 



---



### 4b. Better instructions and a backup
The prompt now tells the LLM to use only the retrieved explanations and to reply "NO MATCH" if none fits the clause. In that case, a short general explanation of the clause type is shown instead, based on the type definitions in Lippi et al. (2019). Tips are limited to realistic actions for users who cannot negotiate the terms.

In [ ]:
EXPLAIN_PROMPT = """You help everyday users understand unfair clauses in online Terms of Service.

Clause: {clause}
Type: {clause_type}

Expert explanations:
{sources}

Rules:
- Use ONLY the expert explanations above. Do not add other legal facts.
- If none of the expert explanations matches what this clause says, reply only: NO MATCH
- Users cannot negotiate these terms, so only suggest realistic actions, such as: checking for an opt-out option, keeping screenshots or records, backing up their content, comparing other services, or contacting a consumer protection organization.
- Do not give legal advice.
- Users cannot negotiate these terms, so only suggest realistic actions, such as: keeping screenshots or records, backing up their content, comparing other services, or contacting a consumer protection organization. Only mention an opt-out if the clause itself mentions one.

Answer in exactly this format:
Why this may be unfair: <2 short sentences in plain English>
What you can do: <1 or 2 short tips>"""

# Backup: one general sentence per type, based on the type definitions in Lippi et al. (2019)
BACKUP = {
    "Arbitration": "the clause requires disputes to be settled by an arbitrator instead of a court.",
    "Unilateral change": "the clause lets the provider change the terms or the service on its own.",
    "Content removal": "the clause lets the provider remove the user's content on its own.",
    "Limitation of liability": "the clause limits or excludes the provider's responsibility for harm or losses.",
    "Unilateral termination": "the clause lets the provider suspend or close the user's account on its own.",
    "Jurisdiction": "the clause decides which courts handle disputes, which may be far from where the user lives.",
    "Choice of law": "the clause decides which country's law applies, which may not be the user's own.",
    "Contract by using": "the clause says that simply using the service means accepting the terms.",
}

def explain(clause, clause_type):
    """Retrieve 3 expert explanations and let the LLM explain from them; use the backup if they don't fit."""
    found = retrieve(clause, clause_type, k=3)
    sources = "\n".join(f"- {t}" for t in found["text"])
    answer = ask_llm(EXPLAIN_PROMPT.format(clause=clause, clause_type=clause_type, sources=sources))
    if "NO MATCH" in answer.upper():
        answer = (f"Why this may be unfair: {BACKUP[clause_type][0].upper()}{BACKUP[clause_type][1:]}\n"
                  "(General explanation for this type: no closely matching expert explanation was found.)")
        return answer, found, True
    return answer, found, False

for clause, ctype in tests:
    answer, found, used_backup = explain(clause, ctype)
    print(f"===== {ctype} {'(BACKUP)' if used_backup else ''} =====\n{answer}")
    print("Sources:", ", ".join(found["name"]), "\n")

===== Unilateral termination  =====
Why this may be unfair: This clause allows Tinder to terminate your account without notice if they believe you have violated the agreement, which means you have no opportunity to contest the decision or correct any perceived violation.

What you can do: Keep screenshots or records of any interactions with Tinder and consider comparing similar services to find one with more transparent policies.
Sources: over_quota, dormant, gen_rights_violation 

===== Arbitration  =====
Why this may be unfair: By waiving your right to a jury trial and agreeing to binding arbitration, you may have less control over how your dispute is resolved, and the process might be more expensive and less transparent than going to court.

What you can do: Check if there is an opt-out provision in the clause, keep screenshots or records of your interactions, and consider comparing similar services that may offer better dispute resolution options.
Sources: arb_obligatory, soft_redi

With the improved prompt, all explanations follow the required format and suggest only realistic actions. Arbitration and Jurisdiction explanations stay within the retrieved sources. For Unilateral termination, the LLM used a partly matching source and added one point not found in the sources, which shows that grounding is not guaranteed; the demo therefore displays the sources next to each explanation.



---



## 5. Hand check
Runs the full RAG on one real test clause per unfair type, chosen from clauses the final Legal-BERT model (run A) correctly flagged. Each explanation is judged on three questions: does it stay within the sources, is it easy to understand, and are the tips realistic.

In [ ]:
from datasets import load_dataset
from google.colab import drive

drive.mount("/content/drive")
test = load_dataset("coastalcph/lex_glue", "unfair_tos")["test"]
names = test.features["labels"].feature.names

logits = np.load("/content/drive/MyDrive/fine_print_detector/predictions/test_logits_A_seed42.npy")
pred = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)

for t, type_name in enumerate(names):
    # First test clause that experts labeled with this type AND the model flagged
    i = next(i for i, labs in enumerate(test["labels"]) if t in labs and pred[i, t] == 1)
    clause = test["text"][i]
    answer, found, used_backup = explain(clause, type_name)
    print(f"===== {type_name} {'(BACKUP)' if used_backup else ''} =====")
    print("CLAUSE:", clause[:300])
    print("SOURCES:", ", ".join(found["name"]))
    print(answer, "\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
===== Limitation of liability  =====
CLAUSE: neither academia.edu nor any other person or entity involved in creating , producing , or delivering the site , services or collective content will be liable for any incidental , special , exemplary or consequential damages , or lost profits , loss of data or loss of goodwill , service interruption 
SOURCES: thirdparty, security, anyloss
Why this may be unfair: This clause limits the provider's responsibility for various types of damages and losses, which can leave users without recourse for issues that arise from using the service, such as data loss or financial losses due to unauthorized activities.

What you can do: Keep detailed records of your interactions and any content you upload. Consider comparing similar services to find one with more favorable terms. If an opt-out option is available, consider taking ad

On one correctly flagged test clause per type, the retrieval found relevant expert explanations for all 8, and the backup was not needed. All 8 explanations were clear, and 7 of 8 gave realistic tips (after updating the prompt to mention an opt-out only when the clause offers one). In 6 of 8, the explanation stayed within the sources; in 2, the LLM added a small point of its own, which is why the demo shows the sources next to each explanation.



---



## 6. Saving the knowledge base
Saves the knowledge base (87 explanations with their type, severity and source) to Google Drive, so the demo can load it.

In [ ]:
save_dir = "/content/drive/MyDrive/fine_print_detector"
kb.to_csv(f"{save_dir}/knowledge_base.csv", index=False)
print("Saved:", len(kb), "explanations to Drive.")

Saved: 87 explanations to Drive.
